## Import Packages

In [1]:
import gcsfs
import fsspec
import warnings
import numpy as np
import xarray as xr
import planetary_computer
from datetime import datetime
import pystac_client as pystac
warnings.filterwarnings('ignore')

## User-Defined Fields

In [60]:
AUTHOR   = 'Savannah L. Ferretti'
EMAIL    = 'savannah.ferretti@uci.edu'
SAVEDIR  = '/ocean/projects/atm200007p/sferrett/Repos/monsoon-pr/data/raw'
YEARS    = [2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014]
MONTHS   = [6,7,8]
LATRANGE = (5.,25.) 
LONRANGE = (60.,90.)
LEVRANGE = (500.,1000.)
FREQUENCY = 'H'

## Functions

In [62]:
def get_era5():
    url  = 'gs://gcp-public-data-arco-era5/ar/1959-2022-full_37-1h-0p25deg-chunk-1.zarr-v2/'
    data = xr.open_zarr(url,decode_times=True)
    data = data.rename({'latitude':'lat','longitude':'lon','level':'lev'})    
    return data

def get_imerg():
    url = 'https://planetarycomputer.microsoft.com/api/stac/v1'
    catalog = pystac.Client.open(url,modifier=planetary_computer.sign_inplace)
    assets  = catalog.get_collection('gpm-imerg-hhr').assets['zarr-abfs']
    data    = xr.open_zarr(fsspec.get_mapper(assets.href,**assets.extra_fields['xarray:storage_options']),consolidated=True)
    return data

def standardize(data):
    dims = ['time','lat','lon','lev'] if 'lev' in data.dims else ['time','lat','lon']
    for dim in dims:
        if dim == 'time' and data.coords[dim].dtype.kind != 'M':
            data.coords[dim] = data.indexes[dim].to_datetimeindex()
        elif dim != 'time':
            data.coords[dim] = data.coords[dim].astype(float)
    data = data.sortby(dims).transpose(*dims)
    return data

def subset(data,years=YEARS,months=MONTHS,latrange=LATRANGE,lonrange=LONRANGE,levrange=LEVRANGE):
    data = data.sel(time=(data['time.year'].isin(self.years))&(data['time.month'].isin(self.months)))
    data = data.sel(lat=slice(*self.latrange),lon=slice(*self.lonrange))
    if 'lev' in ds.dims:
        data = data.sel(lev=slice(*self.levrange))
    return data
    
def resample(data,frequency=FREQUENCY):
    data.coords['time'] = data.time.dt.floor(frequency) 
    return data.groupby('time').first()

def preprocess(data,years=YEARS,months=MONTHS,latrange=LATRANGE,lonrange=LONRANGE,levrange=LEVRANGE,frequency=FREQUENCY):
    data = standardize(data)
    data = subset(years,months,latrange,lonrange,levrange)
    if xr.infer_freq(data.time) != frequency:
        data = resample(data,frequency)
    return data

def dataset(data,shortname,longname,units,source,author=AUTHOR,email=EMAIL):
    vardata = {shortname:([*data.dims],data.data)}
    coords  = {dim: data.coords[dim].data for dim in data.dims}
    ds = xr.Dataset(vardata,coords)
    ds[shortname].attrs = dict(long_name=longname,units=units)
    ds.time.attrs = dict(long_name='Time')
    ds.lat.attrs  = dict(long_name='Latitude',units='°N')
    ds.lon.attrs  = dict(long_name='Longitude',units='°E')
    if 'lev' in data.dims:
        ds.lev.attrs = dict(long_name='Pressure level',units='hPa')
    ds.attrs = dict(source=source,history=f'Created on {datetime.today().strftime("%Y-%m-%d")} by {author} ({email})')
    return ds

def save(data,savedir=SAVEDIR):
    filename = f'OBS_{list(data.keys())[0]}.nc'
    return data.compute().to_netcdf(f'{SAVEDIR}/{filename}.nc')

## Import Data

In [15]:
era5  = get_era5()
imerg = get_imerg()

## New Way

In [77]:
variables = [
    (era5.specific_humidity,'q','Specific humidity','kg/kg','ERA5'),
    (era5.temperature,'t','Air temperature','K','ERA5'),
    (era5.surface_pressure/100,'ps','Surface pressure','hPa','ERA5'),
    (imerg.precipitationCal,'pr','Precipitation flux','mm/hr','IMERG V06')
]

In [ ]:
@dask.delayed
def process(data,varname,longname,units,source,
            years=YEARS,months=MONTHS,latrange=LATRANGE,lonrange=LONRANGE,levrange=LEVRANGE,frequency=FREQUENCY):
    processed = preprocess(data,latrange,lonrange,levrange,frequency)
    ds = dataset(processed,varname,longname,units,source)
    save(ds)

tasks = [process(data,shortname,longname,units,source) for data,shortname,longname,units,source in variables]
dask.compute(*tasks)

## Old Way

In [ ]:
q = era5.specific_humidity
q = subset(q)
q = resample(q,frequency='6-hourly')
q = dataset(q,'q','Specific humidity','kg/kg','ERA5','6-hourly')
save(q,gridtype='HR') # 1h 45m 54s

In [ ]:
t = era5.temperature
t = subset(t)
t = resample(t,frequency='6-hourly')
t = dataset(t,'t','Air temperature','K','ERA5','6-hourly')
save(t,gridtype='HR') # 1h 49m 20s

In [ ]:
ps = era5.surface_pressure/100
ps = subset(ps)
ps = resample(ps,frequency='6-hourly')
ps = dataset(ps,'ps','Surface pressure','hPa','ERA5','6-hourly')
save(ps,gridtype='HR') # 3m 40s

In [ ]:
pr = imerg.precipitationCal*24
pr = subset(pr.where(pr>=0,0))
pr = regrid(pr,resolution=0.25)
pr = resample(pr,'3-hourly mean')
pr = dataset(pr,'pr','Precipitation flux','mm/day','IMERG V06','3-hourly mean')
save(pr,gridtype='HR') # 15m 14s

In [ ]:
import gcsfs
import fsspec
import warnings
import numpy as np
import xarray as xr
import dask
import dask.array as da
import planetary_computer
from datetime import datetime
import pystac_client as pystac
warnings.filterwarnings('ignore')

AUTHOR    = 'Savannah L. Ferretti'
EMAIL     = 'savannah.ferretti@uci.edu'
SAVEDIR   = '/ocean/projects/atm200007p/sferrett/Repos/monsoon-pr/data/raw'
YEARS     = [2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012, 2013, 2014]
MONTHS    = [6, 7, 8]
LATRANGE  = (5., 25.)
LONRANGE  = (60., 90.)
LEVRANGE  = (500., 1000.)
FREQUENCY = 'h'

def get_era5():
    url  = 'gs://gcp-public-data-arco-era5/ar/1959-2022-full_37-1h-0p25deg-chunk-1.zarr-v2/'
    data = xr.open_zarr(url,decode_times=True)
    data = data.rename({'latitude':'lat','longitude':'lon','level':'lev'})
    return data

def get_imerg():
    url = 'https://planetarycomputer.microsoft.com/api/stac/v1'
    catalog = pystac.Client.open(url,modifier=planetary_computer.sign_inplace)
    assets  = catalog.get_collection('gpm-imerg-hhr').assets['zarr-abfs']
    data = xr.open_zarr(fsspec.get_mapper(assets.href, **assets.extra_fields['xarray:storage_options']),consolidated=True)
    return data

def standardize_dims(data):
    dims = ['time','lat','lon','lev'] if 'lev' in data.dims else ['time','lat','lon']
    for dim in dims:
        if dim == 'time' and data.coords[dim].dtype.kind != 'M':
            data.coords[dim] = data.indexes[dim].to_datetimeindex()
        elif dim != 'time':
            data.coords[dim] = data.coords[dim].astype(float)
    return data.sortby(dims).transpose(*dims)

def subset_region(data,latrange=LATRANGE,lonrange=LONRANGE):
    return data.sel(lat=slice(*latrange),lon=slice(*lonrange))

def subset_time(data,years=YEARS,months=MONTHS):
    return data.sel(time=(data['time.year'].isin(years)) & (data['time.month'].isin(months)))

def subset_levels(data,levrange=LEVRANGE):
    return data.sel(lev=slice(*levrange))

def resample_time(data,frequency=FREQUENCY):
    data.coords['time'] = data.time.dt.floor(frequency)
    return data.groupby('time').first()

def preprocess(data, years=YEARS, months=MONTHS, latrange=LATRANGE, lonrange=LONRANGE, levrange=LEVRANGE, frequency=FREQUENCY):
    data = standardize_dims(data)
    data = subset_region(data, latrange, lonrange)
    data = subset_time(data, years, months)
    if 'lev' in data.dims:
        data = subset_levels(data, levrange)
    if xr.infer_freq(data.time) != frequency:
        data = resample_time(data, frequency)
    return data

def dataset(data, varname, longname, units, source, author=AUTHOR, email=EMAIL):
    vardata = {varname: ([*data.dims], data.data)}
    coords = {dim: data.coords[dim].data for dim in data.dims}
    ds = xr.Dataset(vardata, coords)
    ds[varname].attrs = dict(long_name=longname, units=units)
    ds.time.attrs = dict(long_name='Time')
    ds.lat.attrs = dict(long_name='Latitude', units='°N')
    ds.lon.attrs = dict(long_name='Longitude', units='°E')
    if 'lev' in data.dims:
        ds.lev.attrs = dict(long_name='Pressure level', units='hPa')
    ds.attrs = dict(source=source, history=f'Created on {datetime.today().strftime("%Y-%m-%d")} by {author} ({email})')
    return ds

def save(data, savedir=SAVEDIR):
    filename = f'OBS_{list(data.keys())[0]}.nc'
    data.to_netcdf(f'{savedir}/{filename}', compute=True)

era5 = get_era5()
imerg = get_imerg()

variables = [
    (era5.specific_humidity, 'q', 'Specific humidity', 'kg/kg', 'ERA5', FREQUENCY, LATRANGE, LONRANGE, LEVRANGE),
    (era5.temperature, 't', 'Air temperature', 'K', 'ERA5', FREQUENCY, LATRANGE, LONRANGE, LEVRANGE),
    (era5.surface_pressure / 100, 'ps', 'Surface pressure', 'hPa', 'ERA5', FREQUENCY, LATRANGE, LONRANGE, LEVRANGE),
    (imerg.precipitationCal * 24, 'pr', 'Precipitation flux', 'mm/day', 'IMERG V06', '3H', LATRANGE, LONRANGE, None)  # Special frequency and units for IMERG
]

